# ollamas — Colab local-runtime dev loop

Runs the full ollamas **dev + quality loop on the M4 at $0** via local Ollama, from a Colab
notebook connected to the local Docker runtime. Repo is mounted at `/content/ollamas`.

## 1. Environment + reachability

In [ ]:
import os, sys, json, urllib.request

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://host.docker.internal:11434")
GATEWAY = "http://host.docker.internal:3000"

def _get(url, timeout=10):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.status, r.read()

print("node:", os.popen("node --version").read().strip())
print("npm :", os.popen("npm --version").read().strip())
print("python:", sys.version.split()[0])
print("--- repo /content/ollamas ---")
print(os.popen("ls /content/ollamas | head -20").read())

ok = True
try:
    _, body = _get(f"{OLLAMA_HOST}/api/tags")
    models = sorted(m["name"] for m in json.loads(body).get("models", []))
    print("Ollama OK -", len(models), "models:", models)
except Exception as e:
    ok = False; print("Ollama FAIL:", e)
try:
    st, _ = _get(f"{GATEWAY}/api/health")
    print("Gateway /api/health:", st)
except Exception as e:
    ok = False; print("Gateway FAIL:", e)

assert ok, "Ollama or gateway unreachable - start the host services before running this notebook"


## 2. Provision Node 24

The Colab image ships Node 20, but ollamas' store uses `node:sqlite` (`DatabaseSync`), stable in
Node 24 (host runs v24.16.0). Without it the store + server-boot tests fail. Pinned to match host.

In [ ]:
import os, subprocess

NODE = "v24.16.0"
prefix = f"/opt/node-{NODE}-linux-x64"
if not os.path.isdir(prefix):
    print("downloading node", NODE, "(amd64, emulated) ...")
    subprocess.run(
        f"curl -fsSL https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz -o /tmp/node.tar.xz "
        f"&& tar -xJf /tmp/node.tar.xz -C /opt",
        shell=True, check=True, executable="/bin/bash")

# persists for every later ! / subprocess cell (same kernel)
os.environ["PATH"] = f"{prefix}/bin:" + os.environ["PATH"]
print("node:", subprocess.run("node --version", shell=True, capture_output=True, text=True, env=os.environ).stdout.strip())
print("npm :", subprocess.run("npm --version", shell=True, capture_output=True, text=True, env=os.environ).stdout.strip())


## 3. Install dependencies

First run is slow under amd64 emulation. `PUPPETEER_SKIP_DOWNLOAD=1` avoids a Chromium download.

In [ ]:
import subprocess, os
DATA = "/content/ollamas/.colab-data"
env = {**os.environ, "PUPPETEER_SKIP_DOWNLOAD": "1", "MISSION_CONTROL_DATA_DIR": DATA, "CI": "1"}
p = subprocess.run("npm ci 2>&1 | tail -15", shell=True, cwd="/content/ollamas",
                   executable="/bin/bash", env=env, capture_output=True, text=True)
print(p.stdout)
print("[npm ci exit", p.returncode, "]")


## 4. Quality gate (lint -> build -> test)

In [ ]:
import subprocess, os
DATA = "/content/ollamas/.colab-data"
def run(cmd):
    print("$", cmd, flush=True)
    env = {**os.environ, "MISSION_CONTROL_DATA_DIR": DATA, "PUPPETEER_SKIP_DOWNLOAD": "1", "CI": "1"}
    p = subprocess.run(cmd, shell=True, cwd="/content/ollamas", executable="/bin/bash",
                       env=env, capture_output=True, text=True)
    tail = "\n".join((p.stdout + p.stderr).splitlines()[-25:])
    print(tail)
    print(f"[exit {p.returncode}] {cmd}", flush=True)
    return p.returncode

lint = run("npm run lint")


In [ ]:
build = run("npm run build")


In [ ]:
# --no-file-parallelism: run suites sequentially so the amd64-emulated runtime
# (limited CPU/RAM) does not get swamped by many parallel server-boot e2e tests.
test = run("npm test -- --no-file-parallelism")
print({"lint": lint, "build": build, "test": test})
assert lint == 0 and build == 0, "lint/build must pass"


## 5. Tier-0 local-model code review ($0, M4 GPU)

ollamas' own local-LLM dev workflow: a real repo file reviewed by a local model via Ollama, no
cloud cost. Uses `qwen3:8b` with `think:false` for a direct (non-reasoning) answer.

In [ ]:
import urllib.request, json, os
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://host.docker.internal:11434")

target = "/content/ollamas/server/store/db-adapter.ts"
if not os.path.exists(target):
    target = "/content/ollamas/server.ts"
with open(target) as f:
    excerpt = "".join(f.readlines()[:80])
print("Reviewing:", target, f"({excerpt.count(chr(10))} lines)\n")

prompt = ("You are a senior TypeScript reviewer. Review this excerpt from the ollamas project and "
          "list up to 3 concrete, high-value improvements (security, correctness, or clarity). "
          "Be specific.\n\n```ts\n" + excerpt + "\n```")
body = json.dumps({"model": "qwen3:8b", "prompt": prompt, "stream": False,
                   "think": False, "options": {"num_predict": 500, "temperature": 0.2}}).encode()
try:
    req = urllib.request.Request(OLLAMA_HOST + "/api/generate", body, {"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as r:
        d = json.load(r)
    print(d.get("response", "") or "(empty response)")
    print(f"\n[eval_count={d.get('eval_count')} done_reason={d.get('done_reason')}]")
except Exception as e:
    print("local review FAIL:", e)


## 6. Gateway smoke

In [ ]:
import urllib.request, json
GATEWAY = "http://host.docker.internal:3000"

def show(path):
    try:
        with urllib.request.urlopen(GATEWAY + path, timeout=10) as r:
            print(f"GET {path} -> {r.status}")
            print(json.dumps(json.loads(r.read()), indent=2)[:700])
    except urllib.error.HTTPError as e:
        note = " (admin-gated - 401 expected, auth working)" if e.code == 401 else ""
        print(f"GET {path} -> {e.code}{note}")
    except Exception as e:
        print(f"GET {path} -> FAIL {e}")

show("/api/health")
show("/api/models/ollama-local")


## Done

Green env + Node 24 + lint/build/test pass + a local review paragraph + gateway 200 == Colab is a
working, $0 ollamas dev environment on the M4.